calculate proximity

In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import os
import re

import sys
# sys.path.append('./utils/')

from toolbox import wrappers
from toolbox import network_utilities
# from guney_code import network_utils
#import testdisease

# import separation as tools
from multiprocessing import Pool

GenRev not found, steiner wont work
Import error: Negex. Using keyword matching instead
Import error: Funcassociate. Make sure that funcassociate is in toolbox!


In [2]:
import sys
sys.path.append('../FUNCTIONS')
from methods import module_to_json

In [42]:
!pwd
%cd Cell/Reciprocal_reciprocal
!pwd

/home/jovyan/work/drug_screening/Cell/Nlog_reciprocal
/home/jovyan/work/drug_screening/Cell/Reciprocal_nlog
/home/jovyan/work/drug_screening/Cell/Reciprocal_nlog


In [4]:
def targets_to_dict(file=""):
    # importing the module
    import json
     
    # Opening JSON file
    with open(file, 'r') as json_file:
        data = json.load(json_file)
        return data  

In [43]:
target_genes = targets_to_dict(file="../../../transitional_set/Cell/Reciprocal_nlog/target_genes.json")
target_genes

{'up_targets_I_4S': ['PTPN20'],
 'up_targets_III_4S': ['RPS3'],
 'up_targets_IV_4S': ['CCNA2',
  'KIF20A',
  'KIF18B',
  'CENPE',
  'RRM2',
  'TOP2A',
  'CDC6',
  'CENPA',
  'E2F2',
  'PIF1',
  'CENPW',
  'UBE2C',
  'KIF4A',
  'DLGAP5',
  'CDKN3',
  'FOXM1',
  'GINS2',
  'ABCC4',
  'UHRF1',
  'BIRC5',
  'ANP32E',
  'CDCA5',
  'DHFR',
  'TRIM59',
  'FAM72D',
  'PPP2R2B',
  'LAMP5',
  'SIX3-AS1',
  'SPC24',
  'ADCY5',
  'SOX5',
  'MEX3A',
  'B3GNT5',
  'NAV1'],
 'up_targets_III_I': ['ARHGAP24',
  'NIPAL2',
  'NR2C2AP',
  'HMGB3',
  'HYLS1',
  'CLCN4',
  'TRAP1'],
 'up_targets_IV_I': ['PRR11', 'S100A8', 'SCML1'],
 'up_targets_IV_III': ['DHCR24',
  'LRTM1',
  'C9orf106',
  'ATP2A1-AS1',
  'CCL3',
  'MMP19',
  'PLAUR']}

In [6]:
def read_drug_targets(path):
    import pickle
    # Assuming 'data' contains the serialized data
    with open(path, 'rb') as file:
        serialized_data = file.read()
    # Deserialize the data using pickle
    nested_dict = pickle.loads(serialized_data)
    return nested_dict

In [7]:
drug_targets = read_drug_targets(path='../../../drug_screening/proximity/data/target/drug_to_geneids.pcl.all')
len(drug_targets)

5347

In [8]:
import requests

def symbols_to_entrez(symbols):
    base_url = 'https://mygene.info/v3/query'
    entrez_ids = []
    for symbol in symbols:
        params = {'q': f'symbol:{symbol}', 'species': 'human'}
        response = requests.get(base_url, params=params)
        data = response.json()
        if data['total'] > 0:
            try:
                entrez_ids.append(data['hits'][0]['entrezgene'])
            except:
                pass
        
    return entrez_ids

In [44]:
up_targets_I_4S = target_genes["up_targets_I_4S"]
up_targets_I_4S_id = symbols_to_entrez(up_targets_I_4S)
up_targets_III_4S = target_genes["up_targets_III_4S"]
up_targets_III_4S_id = symbols_to_entrez(up_targets_III_4S)
up_targets_IV_4S = target_genes["up_targets_IV_4S"]
up_targets_IV_4S_id = symbols_to_entrez(up_targets_IV_4S)
up_targets_III_I = target_genes["up_targets_III_I"]
up_targets_III_I_id = symbols_to_entrez(up_targets_III_I)
up_targets_IV_I = target_genes["up_targets_IV_I"]
up_targets_IV_I_id = symbols_to_entrez(up_targets_IV_I)
up_targets_IV_III = target_genes["up_targets_IV_III"]
up_targets_IV_III_id = symbols_to_entrez(up_targets_IV_III)

In [46]:
# all symbols are converted to entrez_ids
print(len(up_targets_I_4S))
print(len(up_targets_III_4S))
print(len(up_targets_IV_4S))
print(len(up_targets_III_I))
print(len(up_targets_IV_I))
print(len(up_targets_IV_III))

1
1
34
7
3
7


In [47]:
target_genes_id = dict(
    up_targets_I_4S_id=up_targets_I_4S_id,
    up_targets_III_4S_id=up_targets_III_4S_id,
    up_targets_IV_4S_id=up_targets_IV_4S_id,
    up_targets_III_I_id=up_targets_III_I_id,
    up_targets_IV_I_id=up_targets_IV_I_id,
    up_targets_IV_III_id=up_targets_IV_III_id
    )
target_genes_id

{'up_targets_I_4S_id': ['26095'],
 'up_targets_III_4S_id': ['6188'],
 'up_targets_IV_4S_id': ['890',
  '10112',
  '146909',
  '1062',
  '6241',
  '7153',
  '990',
  '1058',
  '1870',
  '80119',
  '387103',
  '11065',
  '24137',
  '9787',
  '1033',
  '2305',
  '51659',
  '10257',
  '29128',
  '332',
  '81611',
  '113130',
  '1719',
  '286827',
  '728833',
  '5521',
  '24141',
  '100506108',
  '147841',
  '111',
  '6660',
  '92312',
  '84002',
  '89796'],
 'up_targets_III_I_id': ['83478',
  '79815',
  '126382',
  '3149',
  '219844',
  '1183',
  '10131'],
 'up_targets_IV_I_id': ['55771', '6279', '6322'],
 'up_targets_IV_III_id': ['1718',
  '57408',
  '100289092',
  '6348',
  '4327',
  '5329']}

In [48]:
module_to_json(module_dict=target_genes_id, file_path="target_genes_id.json")

In [49]:
target_genes_id = targets_to_dict(file="target_genes_id.json")
target_genes_id

{'up_targets_I_4S_id': ['26095'],
 'up_targets_III_4S_id': ['6188'],
 'up_targets_IV_4S_id': ['890',
  '10112',
  '146909',
  '1062',
  '6241',
  '7153',
  '990',
  '1058',
  '1870',
  '80119',
  '387103',
  '11065',
  '24137',
  '9787',
  '1033',
  '2305',
  '51659',
  '10257',
  '29128',
  '332',
  '81611',
  '113130',
  '1719',
  '286827',
  '728833',
  '5521',
  '24141',
  '100506108',
  '147841',
  '111',
  '6660',
  '92312',
  '84002',
  '89796'],
 'up_targets_III_I_id': ['83478',
  '79815',
  '126382',
  '3149',
  '219844',
  '1183',
  '10131'],
 'up_targets_IV_I_id': ['55771', '6279', '6322'],
 'up_targets_IV_III_id': ['1718',
  '57408',
  '100289092',
  '6348',
  '4327',
  '5329']}

In [14]:
network = wrappers.get_network(network_file="../../../drug_screening/proximity/data/network/network.sif", only_lcc = True)
# Shrinking network to its LCC 11 15
# Final shape: 11 15

In [28]:
len(network.nodes)

13329

In [16]:
list(network.nodes)[:5]

['28988', '118460', '9925', '2044', '23214']

In [17]:
def significant_proximal_drugs(z_score, disease_ids, file):
    drug_proximity = list()
    nodes_to = list(set(disease_ids) & set(network.nodes))
    print(len(nodes_to))
    
    for key in drug_targets:
        nodes_from = list(set(drug_targets[key]) & set(network.nodes))
        try: 
            # Calculate proximity using default measure ("closest") 
            d, z, (mean, sd) = wrappers.calculate_proximity(network, nodes_from, nodes_to, min_bin_size = 100, seed=123456)
            if z < z_score:
                drug_proximity.append((key, z))
        except:
            pass
        
    significant_drugs = sorted(drug_proximity, key=lambda x: x[1])
    with open(file, "w") as f:
        for i in significant_drugs:
            f.write(i[0] + "," + str(i[1]) + "\n")

In [50]:
significant_proximal_drugs(z_score=-1.96, disease_ids=target_genes_id['up_targets_I_4S_id'], file="I_to_4S_-1.96_drugs.txt")

0


In [51]:
significant_proximal_drugs(z_score=-1.96, disease_ids=target_genes_id['up_targets_III_4S_id'], file="III_to_4S_-1.96_drugs.txt")

1


In [ ]:
significant_proximal_drugs(z_score=-1.96, disease_ids=target_genes_id['up_targets_IV_4S_id'], file="IV_to_4S_-1.96_drugs.txt")

25


In [ ]:
significant_proximal_drugs(z_score=-1.96, disease_ids=target_genes_id['up_targets_III_I_id'], file="III_to_I_-1.96_drugs.txt")

In [ ]:
significant_proximal_drugs(z_score=-1.96, disease_ids=target_genes_id['up_targets_IV_I_id'], file="IV_to_I_-1.96_drugs.txt")

In [ ]:
significant_proximal_drugs(z_score=-1.96, disease_ids=target_genes_id['up_targets_IV_III_id'], file="IV_to_III_-1.96_drugs.txt")

see what genes are drug targets in the network

In [ ]:
network_genes = set(network.nodes)

In [ ]:
I_to_4S_disease_genes = set(target_genes_id['up_targets_I_4S_id']) & network_genes
I_to_4S_disease_genes

In [ ]:
III_to_4S_disease_genes = set(target_genes_id['up_targets_III_4S_id']) & network_genes
III_to_4S_disease_genes

In [ ]:
IV_to_4S_disease_genes = set(target_genes_id['up_targets_IV_4S_id']) & network_genes
IV_to_4S_disease_genes

In [ ]:
III_to_I_disease_genes = set(target_genes_id['up_targets_III_I_id']) & network_genes
III_to_I_disease_genes

In [ ]:
IV_to_I_disease_genes = set(target_genes_id['up_targets_IV_I_id']) & network_genes
IV_to_I_disease_genes

In [ ]:
IV_to_III_disease_genes = set(target_genes_id['up_targets_IV_III_id']) & network_genes
IV_to_III_disease_genes